In [43]:
import os

import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent

In [44]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")



In [45]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0,api_key=GEMINI_API_KEY)

In [46]:
response = llm.invoke("what is the current year")

response

AIMessage(content='The current year is **2024**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--01a1171f-a49f-74d2-8e2a-0ec07fce7570-0', usage_metadata={'input_tokens': 6, 'output_tokens': 610, 'total_tokens': 616, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 600}})

In [47]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful research assistant. Use your search tool for anything current, then answer concisely."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])


In [48]:
search_tool = TavilySearchResults(max_results=3, tavily_api_key=TAVILY_API_KEY)

In [56]:
@tool
def get_weather_data(city: str) -> str:
    """Get the current weather for a city.

    Use this for any question about current weather, temperature,
    or conditions in a specific place.

    Args:
        city: The city to retrieve weather for, e.g. "Hyderabad".
    """
    # Free Weatherstack plans are HTTP-only; https 403s on them.
    resp = requests.get(
        "http://api.weatherstack.com/current",
        params={"access_key": WEATHERSTACK_API_KEY, "query": city},
        timeout=10,
    )
    resp.raise_for_status()
    data = resp.json()

    # Weatherstack signals failure in the body with a 200 status.
    if not data.get("success", True):
        return f"Weather lookup failed: {data.get('error', {}).get('info', 'unknown error')}"

    if "current" not in data:
        return f"No weather data found for {city!r}. Check the city name and try again."

    current = data["current"]
    return (
        f"City: {data.get('location', {}).get('name', city)}"
        f"Temperature: {current['temperature']}°C"
        f"Weather: {current['weather_descriptions'][0]}"
        f"Humidity: {current['humidity']}%"
    )

In [57]:
tools = [search_tool, get_weather_data]

In [58]:
agent = create_tool_calling_agent(llm, tools, prompt=prompt)

In [59]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [60]:
response = agent_executor.invoke({"input": "what is india's capital? and its wheather and humidity?"})

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': "India's capital"}`


[{'title': "Where Is India's Capital: New Delhi's Location and ...", 'url': 'https://iotserver4g.trumeter.com/iot/where-is-indias-capital-new-delhis-location-and-significance-tfm7m6', 'content': 'New Delhi is the capital of India, situated in the north-central region of the country along the banks of the Yamuna River. It serves as the primary seat of', 'score': 0.87782305}, {'title': 'List of capitals of India - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_India', 'content': '## Modern period\n\n[edit]\n\n In 1858, Allahabad the capital of India for a day when it also served as the capital of North-Western Provinces.\n During the British Raj, until 1911, Calcutta was the capital of India.\n By the latter half of the 19th century, Shimla had become the summer capital.\n King George V proclaimed the transfer of the capital from Calcutta to Delhi a